In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.config import RAW_OPENAQ_DIR, INTERIM_DIR, PROCESSED_DIR

# Load raw data
raw_df = pd.read_parquet(RAW_OPENAQ_DIR / "jakarta_raw_combined.parquet")
print(f"Raw records: {len(raw_df):,}")
print(f"Columns: {raw_df.columns.tolist()}")

Raw records: 2,219
Columns: ['value', 'coordinates', 'parameter_name', 'flagInfo.hasFlags', 'parameter.id', 'parameter.name', 'parameter.units', 'parameter.displayName', 'period.label', 'period.interval', 'period.datetimeFrom.utc', 'period.datetimeFrom.local', 'period.datetimeTo.utc', 'period.datetimeTo.local', 'summary.min', 'summary.q02', 'summary.q25', 'summary.median', 'summary.q75', 'summary.q98', 'summary.max', 'summary.avg', 'summary.sd', 'coverage.expectedCount', 'coverage.expectedInterval', 'coverage.observedCount', 'coverage.observedInterval', 'coverage.percentComplete', 'coverage.percentCoverage', 'coverage.datetimeFrom.utc', 'coverage.datetimeFrom.local', 'coverage.datetimeTo.utc', 'coverage.datetimeTo.local', 'station_name', 'location_id']


In [2]:
# Kolom yang kita butuhkan dari raw
clean_df = pd.DataFrame({
    "datetime_utc"   : raw_df["period.datetimeFrom.utc"],
    "station_name"   : raw_df["station_name"],
    "location_id"    : raw_df["location_id"],
    "pm25_mean"      : raw_df["summary.avg"],
    "pm25_median"    : raw_df["summary.median"],
    "pm25_max"       : raw_df["summary.max"],
    "pm25_min"       : raw_df["summary.min"],
    "pm25_p02"       : raw_df["summary.q02"],
    "pm25_p98"       : raw_df["summary.q98"],
    "coverage_pct"   : raw_df["coverage.percentCoverage"],
    "obs_count"      : raw_df["coverage.observedCount"],
})

print(clean_df.head())
print(f"\nShape: {clean_df.shape}")

           datetime_utc   station_name  location_id  pm25_mean  pm25_median  \
0  2022-10-03T17:00:00Z  Jakarta South         8320  27.681818         28.5   
1  2022-10-04T17:00:00Z  Jakarta South         8320  39.000000         39.0   
2  2023-05-06T17:00:00Z  Jakarta South         8320        NaN          NaN   
3  2023-05-07T17:00:00Z  Jakarta South         8320        NaN          NaN   
4  2023-05-08T17:00:00Z  Jakarta South         8320  45.904762         44.0   

   pm25_max  pm25_min  pm25_p02  pm25_p98  coverage_pct  obs_count  
0      42.0       8.0      8.00     41.58          92.0         22  
1      42.0      36.0     36.12     41.88           8.0          2  
2       NaN       NaN       NaN       NaN          71.0         17  
3       NaN       NaN       NaN       NaN          75.0         18  
4      76.0      19.0     23.40     73.20         100.0         24  

Shape: (2219, 11)


In [3]:
# Parse datetime
clean_df["datetime_utc"] = pd.to_datetime(clean_df["datetime_utc"], utc=True)
clean_df["date"] = clean_df["datetime_utc"].dt.tz_convert("Asia/Jakarta").dt.date
clean_df["date"] = pd.to_datetime(clean_df["date"])

print(f"Date range: {clean_df['date'].min()} to {clean_df['date'].max()}")
print(f"Unique dates: {clean_df['date'].nunique()}")
print(f"Unique stations: {clean_df['station_name'].unique()}")

Date range: 2019-01-02 00:00:00 to 2024-01-01 00:00:00
Unique dates: 1766
Unique stations: ['Jakarta South' 'Jakarta Central' 'West Jakarta Mayor Office'
 'East Jakarta Mayor Office' 'Rusunawa Marunda']


In [4]:
print("=== BEFORE CLEANING ===")
print(f"Total records: {len(clean_df):,}")
print(f"PM2.5 mean stats:\n{clean_df['pm25_mean'].describe()}")

# Flag 1: Low coverage days (< 75% data completeness)
low_coverage = clean_df["coverage_pct"] < 75
print(f"\nLow coverage records (<75%): {low_coverage.sum()}")

# Flag 2: Physically impossible values
impossible_high = clean_df["pm25_mean"] > 999
impossible_low  = clean_df["pm25_mean"] < 0
print(f"Impossible high (>999): {impossible_high.sum()}")
print(f"Impossible low (<0): {impossible_low.sum()}")

# Flag 3: Extreme outliers (> 500 ug/m3 — sensor malfunction threshold)
extreme = clean_df["pm25_mean"] > 500
print(f"Extreme values (>500): {extreme.sum()}")

# Apply flags — set to NaN, don't delete rows
clean_df.loc[low_coverage, "pm25_mean"] = np.nan
clean_df.loc[impossible_high | impossible_low | extreme, "pm25_mean"] = np.nan

print(f"\n=== AFTER CLEANING ===")
print(f"Valid PM2.5 records: {clean_df['pm25_mean'].notna().sum():,}")
print(f"NaN records: {clean_df['pm25_mean'].isna().sum():,}")

=== BEFORE CLEANING ===
Total records: 2,219
PM2.5 mean stats:
count    1836.000000
mean       44.853133
std        67.741756
min         0.500000
25%        26.410326
50%        38.354167
75%        49.088542
max       985.000000
Name: pm25_mean, dtype: float64

Low coverage records (<75%): 174
Impossible high (>999): 0
Impossible low (<0): 0
Extreme values (>500): 11

=== AFTER CLEANING ===
Valid PM2.5 records: 1,680
NaN records: 539


In [5]:
# Jakarta Central adalah primary station (coverage 2019-2023)
jakarta_central = clean_df[
    clean_df["station_name"] == "Jakarta Central"
].copy()

# Set date sebagai index
jakarta_central = jakarta_central.sort_values("date").set_index("date")

print(f"Jakarta Central records: {len(jakarta_central):,}")
print(f"Date range: {jakarta_central.index.min()} to {jakarta_central.index.max()}")

# Check missing dates
full_date_range = pd.date_range(start="2019-01-01", end="2023-12-31", freq="D")
missing_dates = full_date_range.difference(jakarta_central.index)
print(f"\nMissing dates: {len(missing_dates)} out of {len(full_date_range)}")
print(f"Coverage: {(1 - len(missing_dates)/len(full_date_range))*100:.1f}%")

Jakarta Central records: 1,766
Date range: 2019-01-02 00:00:00 to 2024-01-01 00:00:00

Missing dates: 61 out of 1826
Coverage: 96.7%


In [6]:
from src.config import PSBB_PERIODS

# Reindex to full date range (fill missing with NaN)
jakarta_central = jakarta_central.reindex(full_date_range)
jakarta_central.index.name = "date"

# Time features
jakarta_central["year"]      = jakarta_central.index.year
jakarta_central["month"]     = jakarta_central.index.month
jakarta_central["dayofweek"] = jakarta_central.index.dayofweek
jakarta_central["quarter"]   = jakarta_central.index.quarter
jakarta_central["season"]    = jakarta_central["month"].apply(
    lambda m: "wet" if m in [11, 12, 1, 2, 3, 4] else "dry"
)

# PSBB flags
jakarta_central["is_psbb"] = False
jakarta_central["psbb_phase"] = "none"

for period in PSBB_PERIODS["jakarta"]:
    mask = (
        (jakarta_central.index >= period["start"]) &
        (jakarta_central.index <= period["end"])
    )
    jakarta_central.loc[mask, "is_psbb"] = True
    jakarta_central.loc[mask, "psbb_phase"] = period["phase"]

print(f"PSBB days flagged: {jakarta_central['is_psbb'].sum()}")
print(f"\nPSBB phase distribution:")
print(jakarta_central["psbb_phase"].value_counts())

PSBB days flagged: 185

PSBB phase distribution:
psbb_phase
none          1641
transition     101
strict          84
Name: count, dtype: int64


In [7]:
# Save interim
interim_path = INTERIM_DIR / "jakarta_central_interim.parquet"
jakarta_central.to_parquet(interim_path)

print(f"Saved to: {interim_path}")
print(f"\nFinal shape: {jakarta_central.shape}")
print(f"\nPM2.5 summary (clean):")
print(jakarta_central["pm25_mean"].describe())
print(f"\nSample:")
jakarta_central[["pm25_mean", "year", "season", "is_psbb", "psbb_phase"]].head(10)

Saved to: C:\AQSEL\me\KARIR\Portofolio Setelah Wisuda\EDA Project Indonesia Urban Air Quality Analysis\data\interim\jakarta_central_interim.parquet

Final shape: (1826, 18)

PM2.5 summary (clean):
count    1355.000000
mean       39.027798
std        34.510310
min         1.000000
25%        23.782197
50%        35.416667
75%        46.899510
max       469.708333
Name: pm25_mean, dtype: float64

Sample:


,pm25_mean,year,season,is_psbb,psbb_phase
date,,,,,
2019-01-01,NaN,2019,wet,False,none
2019-01-02,5.956522,2019,wet,False,none
2019-01-03,7.260870,2019,wet,False,none
2019-01-04,14.041667,2019,wet,False,none
2019-01-05,26.083333,2019,wet,False,none
2019-01-06,39.333333,2019,wet,False,none
2019-01-07,31.916667,2019,wet,False,none
2019-01-08,25.791667,2019,wet,False,none
2019-01-09,29.166667,2019,wet,False,none
